In [1]:
import gym
from custom_env.optical_rl_gym.envs.rmsa_env  import RMSAEnv
from custom_env.optical_rl_gym.utils import evaluate_heuristic
from custom_env.CustomRLenv.model import HierarchicalPolicy
from custom_env.CustomRLenv.CustomRMSAEnv import  CustomRMSAEnv
from custom_env.CustomRLenv.utils import transform_graph
from custom_env.CustomRLenv.utils import get_topology


from env import constant
import pickle
import logging
import numpy as np
import torch
import matplotlib.pyplot as plt


import torch.optim as optim

from collections import defaultdict

class TrainingLogger:
    def __init__(self, env):
        self.stats = defaultdict(list)
        self.env  = env
    def log_step(self, reward, service_blocking_rate, bit_rate_blocking_rate, avg_link_utilization):
        self.stats["reward"].append(reward)
        self.stats["service_blocking_rate"].append(service_blocking_rate)
        self.stats["bit_rate_blocking_rate"].append(bit_rate_blocking_rate)
        self.stats["avg_link_utilization"].append(avg_link_utilization)

    def log_episode(self):
        # Compute episode-level summaries
        self.stats["episode_reward"].append(sum(self.stats["reward"][-self.env.episode_length:]))
        self.stats["episode_service_blocking_rate"].append(np.mean(self.stats["service_blocking_rate"][-self.env.episode_length:]))
        self.stats["episode_bit_rate_blocking_rate"].append(np.mean(self.stats["bit_rate_blocking_rate"][-self.env.episode_length:]))
        self.stats["episode_avg_link_utilization"].append(np.mean(self.stats["avg_link_utilization"][-self.env.episode_length:]))


def compute_gae(rewards,values,dones,gamma=0.99,lam=0.95):

    advantages=[]
    gae=0
    next_value=0

    for t in reversed(range(len(rewards))):

        delta=rewards[t]+gamma*next_value*(1-dones[t])-values[t]

        gae=delta+gamma*lam*(1-dones[t])*gae

        advantages.insert(0,gae)

        next_value=values[t]

    returns=[a+v for a,v in zip(advantages,values)]

    return torch.tensor(advantages),torch.tensor(returns)

def ppo_update(model,optimizer,logprobs_old,logprobs_new,
               advantages,returns,values,clip=0.2):

    ratio=torch.exp(logprobs_new-logprobs_old)

    surr1=ratio*advantages
    surr2=torch.clamp(ratio,1-clip,1+clip)*advantages

    policy_loss=-torch.min(surr1,surr2).mean()

    value_loss=(returns-values).pow(2).mean()

    loss=policy_loss+0.5*value_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

load = 120  # Traffic load, measured in Erlangs
seed = 20  # Seed of environment
episodes = 100 # Number of episodes per execution
episode_length = 300  # Episode Length

# with open(f'optical_rl/examples/topologies/nsfnet_chen_5-paths_6-modulations.h5', 'rb') as f:
#    topology = pickle.load(f)

topology = get_topology(f'./data/germany/sndlib_germany.txt', 'Germany', sndformat=True, alpha=1)
# topology = get_topology('./data/nsf/nsfnet_chen.txt', 'NSFNET')

# Environment arguments for the simulation
env_args = dict(topology=topology, 
                seed=seed, 
                allow_rejection=True, 
                load=load, 
                mean_service_holding_time=200,
                episode_length=episode_length, 
                num_spectrum_resources=30,
                bit_rates = constant.bit_rates,)

# -----------------------------
# Environment setup
# -----------------------------
env = CustomRMSAEnv(**env_args)

# Get environment info
num_nodes = env.num_nodes
num_edges = env.topology.number_of_edges()
num_paths = env.k_paths
num_modulations =  len(env.topology.graph['modulations'])
num_spectra = env.num_spectrum_resources

# -----------------------------
# TODO: Transformed graph: new graph, with each node corresponding to an edge in the original topology
# -----------------------------
transformed_topology = transform_graph(topology)
# -----------------------------
# RL agent setup
# -----------------------------
device = "cpu" #torch.device("cuda" if torch.cuda.is_available() else "cpu")


policy = HierarchicalPolicy(
    node_dim=env.get_node_features().shape[1],         # degree, betweenness
    edge_dim=env.get_edge_features().shape[1],
    hidden_dim=128,
    # num_paths=num_paths,
    num_mod=num_modulations,
    num_slots=num_spectra
).to(device)

optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)


logger = TrainingLogger(env)


# Build edge_index
graph_nx = env.topology
edges =  [tuple(map(int, t)) for t in graph_nx.edges()]
if len(edges) == 0:  # avoid empty graph
    edge_index = torch.zeros((2,0), dtype=torch.long).to(device)
else:
    # nodes are from 1 to num_nodes 
    # we do -1 so that it is now from 0 to num_nodes - 1
    # the library can be updated so that nodes are directly from 0
    edge_index = torch.tensor(edges, dtype=torch.long).T - 1  # [2, num_edges]
        
    

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


	 Path(path_id=0, node_list=['1', '47', '43', '25', '46', '48', '2'], hops=6, length=487, best_modulation=Modulation(name='16QAM', maximum_length=500, spectral_efficiency=4, minimum_osnr=22.4, inband_xt=-23), current_modulation=None)
	 Path(path_id=1, node_list=['1', '47', '43', '24', '25', '46', '48', '2'], hops=7, length=506, best_modulation=Modulation(name='8QAM', maximum_length=1000, spectral_efficiency=3, minimum_osnr=18.6, inband_xt=-20), current_modulation=None)
	 Path(path_id=2, node_list=['1', '30', '29', '24', '25', '46', '48', '2'], hops=7, length=511, best_modulation=Modulation(name='8QAM', maximum_length=1000, spectral_efficiency=3, minimum_osnr=18.6, inband_xt=-20), current_modulation=None)
	 Path(path_id=3, node_list=['1', '30', '29', '17', '10', '34', '25', '46', '48', '2'], hops=9, length=549, best_modulation=Modulation(name='8QAM', maximum_length=1000, spectral_efficiency=3, minimum_osnr=18.6, inband_xt=-20), current_modulation=None)
	 Path(path_id=4, node_list=['1', 

In [2]:
state = env.customreset(False)
done = False

logps=[]
values=[]
rewards=[]
dones=[]

while not done:
    
    # Convert features to torch tensors
    node_features = torch.tensor(state["node_features"], 
                                    dtype=torch.float32).to(device)
    edge_features = torch.tensor(state["edge_features"], 
                                    dtype=torch.float32).to(device)
    demand_embedding = torch.tensor(state["demand_embedding"], 
                                    dtype=torch.float32).unsqueeze(0).to(device)
    path_impairments   = torch.tensor(state['candidate_paths_impairment'], 
                                        dtype=torch.float32).to(device)

    # candidate_paths = torch.tensor(state['candidate_paths'],
    #                                dtype=torch.long).to(device)
    
    candidate_paths = state['candidate_paths']
    path_spectrum = state['path_spectrum']
                                    
    

    
    # Masks (binary tensors)
    path_mask, mod_mask, spec_mask = state["masks"]
    
    path_spectrum = torch.tensor(path_spectrum, dtype=torch.float32).to(device)
    path_mask = torch.tensor(path_mask, dtype=torch.bool).to(device)
    mod_mask = torch.tensor(mod_mask, dtype=torch.bool).to(device)
    spec_mask = torch.tensor(spec_mask, dtype=torch.bool).to(device)
    
    
    
    current_state = dict(
        node_feat=node_features,
        edge_feat=edge_features,
        edge_index=edge_index,
        paths=candidate_paths,
        path_mask=path_mask,
        path_spectrum=path_spectrum,
        path_features=path_impairments,
        mod_masks=mod_mask,
        spec_masks=spec_mask
    )

    
    # Get action + probabilities + value
    action, logprob, value = policy(current_state)
    
    # print(f"action = {action}")

    # Step environment
    next_state, reward, done, info = env.step(action)

    logps.append(logprob)
    values.append(value.item())
    rewards.append(reward)
    dones.append(done)

    
    # new state 
    state = next_state
    
    # Logging
    logger.log_step(reward, 
                    info['service_blocking_rate'],
                    info['bit_rate_blocking_rate'],
                    info['avg_link_utilization']
                    )

logps = torch.stack(logps)

advantages, returns = compute_gae(rewards, values, dones)


#------------------------------------------------------------
#------------------------------------------------------------
# recompute logprobs with current policy
new_logps = []
new_values = []

state = env.customreset(False)

done=False

while not done:
    
    # Convert features to torch tensors
    node_features = torch.tensor(state["node_features"], 
                                    dtype=torch.float32).to(device)
    edge_features = torch.tensor(state["edge_features"], 
                                    dtype=torch.float32).to(device)
    demand_embedding = torch.tensor(state["demand_embedding"], 
                                    dtype=torch.float32).unsqueeze(0).to(device)
    path_impairments   = torch.tensor(state['candidate_paths_impairment'], 
                                        dtype=torch.float32).to(device)

    # candidate_paths = torch.tensor(state['candidate_paths'],
    #                                dtype=torch.long).to(device)
    
    candidate_paths = state['candidate_paths']
    path_spectrum = state['path_spectrum']
                                    
    

    
    # Masks (binary tensors)
    
    path_mask, mod_mask, spec_mask = state["masks"]
    
    path_spectrum = torch.tensor(path_spectrum, dtype=torch.float32).to(device)
    path_mask = torch.tensor(path_mask, dtype=torch.bool).to(device)
    mod_mask = torch.tensor(mod_mask, dtype=torch.bool).to(device)
    spec_mask = torch.tensor(spec_mask, dtype=torch.bool).to(device)
    
    
    
    current_state = dict(
        node_feat=node_features,
        edge_feat=edge_features,
        edge_index=edge_index,
        paths=candidate_paths,
        path_mask=path_mask,
        path_spectrum=path_spectrum,
        path_features=path_impairments,
        mod_masks=mod_mask,
        spec_masks=spec_mask
    )

    
    # Get action + probabilities + value
    action, logprob, value = policy(current_state)
    # Step environment
    next_state, reward, done, info = env.step(action)
    
    new_logps.append(logprob)
    new_values.append(value)
    
    state = next_state
    
new_logps=torch.stack(new_logps)
new_values=torch.stack(new_values)


loss=ppo_update(
    policy,
    optimizer,
    logps.detach(),
    new_logps,
    advantages,
    returns,
    new_values
)

C:\Users\quang\AppData\Local\Temp\ipykernel_22216\1692797613.py:33: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_new.cpp:281.)
  path_spectrum = torch.tensor(path_spectrum, dtype=torch.float32).to(device)


Edge, Numerator, denominator ('1', '30') 47.0 47.0
Edge, Numerator, denominator ('1', '49') 47.0 47.0
Edge, Numerator, denominator ('1', '47') 47.0 47.0
Edge, Numerator, denominator ('2', '48') 47.0 47.0
Edge, Numerator, denominator ('2', '35') 47.0 47.0
Edge, Numerator, denominator ('2', '50') 47.0 47.0
Edge, Numerator, denominator ('3', '32') 47.0 47.0
Edge, Numerator, denominator ('3', '9') 47.0 47.0
Edge, Numerator, denominator ('3', '38') 47.0 47.0
Edge, Numerator, denominator ('4', '32') 47.0 47.0
Edge, Numerator, denominator ('4', '12') 47.0 47.0
Edge, Numerator, denominator ('4', '44') 47.0 47.0
Edge, Numerator, denominator ('4', '33') 47.0 47.0
Edge, Numerator, denominator ('4', '21') 47.0 47.0
Edge, Numerator, denominator ('5', '36') 47.0 47.0
Edge, Numerator, denominator ('5', '45') 47.0 47.0
Edge, Numerator, denominator ('5', '23') 47.0 47.0
Edge, Numerator, denominator ('5', '6') 47.0 47.0
Edge, Numerator, denominator ('6', '33') 47.0 47.0
Edge, Numerator, denominator ('6'

In [ ]:
env.generated_req_lifetime

[(1.8681465468418599, 378.5461034231404),
 (1.9137310281897788, 104.42459418405203),
 (6.907922259306439, 119.34961181079004),
 (7.5722564180069885, 19.461777492160156),
 (7.713883651516812, 170.292611381774),
 (8.2009776076834, 289.149645662775),
 (12.136217921936462, 62.272954599838705),
 (12.588290840964719, 2.6130129138631495),
 (14.566162063906429, 81.85181206855265),
 (15.918214083694513, 113.44428400120712),
 (17.869126263530084, 1104.4090770175376),
 (21.82422051592534, 10.644931133340208),
 (24.873141511763045, 24.39543912040407),
 (26.73240649596015, 102.8649358392531),
 (29.743758706783176, 35.84815451461435),
 (29.87027883201874, 275.0619635313482),
 (35.08359890815212, 303.87571814692086),
 (35.4969235232914, 141.3033141896947),
 (39.27325916219193, 444.9630179846486),
 (40.09173813415999, 49.18060604951672),
 (40.31500442417711, 98.77965662606323),
 (40.32041561719177, 41.7895467222353),
 (44.312110429535615, 70.33246658835475),
 (44.32022044498711, 12.757413760649419),
 

: 